In [6]:
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
from pypdf import PdfReader
from pathlib import Path
from dotenv import load_dotenv
import ollama
import re
import os

In [9]:
env_path = Path(r"..\.env")
load_dotenv(env_path)

EMBED_MODEL = "bge-m3"

In [4]:
text="""Marché à prix au pourcentage :
Un marché est qualifié de marché à prix au pourcentage lorsque la rémunération du titulaire est déterminée
en appliquant un pourcentage au montant hors taxes des travaux effectivement
réalisés et dûment constatés. Ce montant de référence exclut les effets de la révision
des prix ainsi que les éventuelles indemnités et pénalités."""

In [10]:
database_url = os.getenv("DATABASE_URL")
conn = psycopg2.connect(database_url)

In [12]:
def embed_batch(texts):
    response = ollama.embed(model=EMBED_MODEL, input=texts)
    return response["embeddings"]

In [19]:
embed_text=embed_batch(text)[0]

In [23]:
conn.rollback()
cur=conn.cursor()
cur.execute("""
    SELECT id, pdf_name, page_num, chunk_text
    FROM dev.chunks
    ORDER BY embedding <=> %s::vector
    LIMIT 5;
""",(embed_text,))

In [25]:
results = cur.fetchall()
for row in results:
    print(row)
conn.close()